# 1. torchao框架介绍

- 我们正在将所有与量化相关的开发集中到torchao。官网文档地址：
    - `https://docs.pytorch.org/ao/stable/index.html`

- 现有量化流程的计划如下：
    - Eager 模式量化 (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic)：
        - 请迁移使用 torchao 的 eager 模式 quantize_ API 替代。
    - FX 图模式量化 (torch.ao.quantization.quantize_fx.prepare_fx, torch.ao.quantization.quantize_fx.convert_fx)：
        - 请迁移使用 torchao 的 pt2e 量化 API 替代 (torchao.quantization.pt2e.quantize_pt2e.prepare_pt2e, torchao.quantization.pt2e.quantize_pt2e.convert_pt2e)。
    - pt2e 量化：
        - 已迁移到 torchao (`https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e`)，详情请参阅`https://github.com/pytorch/ao/issues/2259`。
    - 注意：如果没有阻塞问题，PyTorch计划在 2.10 版本中删除 torch.ao.quantization，或在所有阻塞问题清除后的最早 PyTorch 版本中执行此操作（截止写这这个文档的时间，torch2.10已经作为稳定版本发布，其中torch.ao大部分已经被移除，但没有全部移除）。

- 环境说明：
    - torch 2.10
    - torchao 0.16

# 2. Eager模式量化

- Eager模式量化已经被quantize_函数替换，包含实现
    - 动态量化（原来的torch.ao.quantization.quantize_dynamic）
    - 静态量化（原来的torch.ao.quantization.quantize）
    - 训练感知量化（原来的torch.ao.quantization.quantize）

## 2.1. 动态量化(训练后动态量化PTDQ)

- 动态量化的主要事项：
    1. 量化适用于以下层类型：
       - nn.Linear
       - nn.Conv1d, nn.Conv2d, nn.Conv3d
       - nn.LSTM, nn.GRU (部分支持)
    2. 动态量化特别适合：
       - NLP模型（如Transformer、BERT）
       - RNN/LSTM模型
       - 批量大小变化大的场景
    3. 量化前后的性能对比：
       - 检查准确率损失（通常<1%是可接受的）
       - 测量推理速度提升
       - 检查内存占用减少
    4. 不适用动态量化的场景：
       - 需要静态量化的CNN模型（考虑静态量化）
       - 对精度极其敏感的任务
       - 模型中有不支持量化的操作

### (1) 核心函数说明

```python
def quantize_(
    model: torch.nn.Module,   # 输入的需要量化的模型
    config: AOBaseConfig,     # 模型的配置
    filter_fn: Optional[Callable[[torch.nn.Module, str], bool]] = _is_linear,  # 一个函数，它接收一个 nn.Module实例及其完全限定名称作为输入，如果希望对该模块应用配置，则返回 True。
    device: Optional[torch.types.Device] = None,   # 在应用 filter_fn之前，将模块移动到的目标设备。可设置为 "cuda" 以加速量化。最终模型将位于指定的设备上。默认值为 None（不改变设备）。
)
```

- config参数可以取值如下：
    - float8权重配置：
        - Float8DynamicActivationFloat8WeightConfig：
            - 用于对线性层的**激活值**和**权重**均应用float8**动态对称量化**的配置。
        - Float8WeightOnlyConfig
            - 用于对线性层应用float8仅**权重量化**、**逐通道对称量化**的配置。
    - int8权重配置：
        - Int8DynamicActivationInt8WeightConfig：
            - 用于对线性层应用int8**动态对称**、**逐词元（per-token）激活值**量化和int8**逐通道权重量化**的配置。
        - Int8WeightOnlyConfig：
            - 用于对线性层应用int8**仅权重量化**、**逐通道对称量化**的配置。
    - int4权重配置：
        - Int4WeightOnlyConfig：
            - 用于int4**仅权重量化**的配置。目前**仅支持分组量化**。提供版本1和版本 2，这两个版本实现方式不同，但功能支持相同。
        - Float8DynamicActivationInt4WeightConfig：
            - 用于对线性层应用float8**动态**、**逐行激活值量化**和int4**逐组权重量化**的配置。（目前仅支持组大小为 128，因为底层内核仅支持 128 及以上，且增大组大小并无益处。）
    - intx权重配置：
        - IntxWeightOnlyConfig：
            - 用于将权重量化为 torch.intx（其中 1 <= x <= 8）的配置。权重使用 weight_dtype 指定的位数，以分组（groupwise）或逐通道（channelwise）方式进行量化，并带有缩放因子和零点。
        - Int8DynamicActivationIntxWeightConfi：
            - 用于将激活值动态量化为torch.int8并将权重量化为torch.intx（其中 1<=x<=8）的配置。
    - mx权重配置：
        - MXDynamicActivationMXWeightConfig：MX 格式推理量化
    - nvfp4权重配置：
        - NVFP4DynamicActivationNVFP4WeightConfig：NVIDIA FP4 (NVFP4) 推理量化配置
        - NVFP4WeightOnlyConfig

### (2) 加载模型 

- 模型可以是用户定制的网络模型，也可以是Pytorch预训练模型，也可以是HuggingFace提供的预训练模型。

In [1]:
def load_model(model_name="alex"):
    import torch
    import torchvision
    if model_name == "alex":
        _model = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.DEFAULT)
        return _model
    if model_name == "vgg":
        _model = torchvision.models.vgg11(weights=torchvision.models.VGG11_Weights.DEFAULT)
        return _model
    if model_name == "resnet":
        _model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        return _model
    if model_name == "vit":
        _model = torchvision.models.vit_b_16(weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
        return _model

In [2]:
model_alex = load_model()
model_alex

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

### (3) 量化模型

In [3]:
def quantize_dynamic(model):
    from torchao.quantization import Int8DynamicActivationInt8WeightConfig, quantize_
    import copy
    _model_w8a8 = copy.deepcopy(model)
    quantize_(_model_w8a8, Int8DynamicActivationInt8WeightConfig())
    return _model_w8a8

In [4]:
model_w8a8 = quantize_dynamic(model_alex)
model_w8a8

W0317 14:13:14.768000 4068 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True, w

### (4) 加载模型验证数据集

In [5]:
# 返回DataLoader对象。
def load_data(root="F:/04Datasets/ImageNet2012", split="val"):
    import torchvision.transforms as transforms
    from torchvision.datasets import ImageNet
    from torch.utils.data import DataLoader
    import torch
    import torchvision
    torch.manual_seed(2026)
    # 加载数据集
    _ds_imagenet2012 = ImageNet(
        root=root,
        split=split,
        transform = torchvision.models.AlexNet_Weights.IMAGENET1K_V1.transforms() # 需要是对象
        # target_transform=None,   # 标签转换
        # loader=Image.open   # 默认（还可以直接加载为Tensor：）
    )
    # 取部分子集
    _num_calibration = 1000   # 总样本是50000 
    _num_calibration = _num_calibration if _num_calibration<=len(_ds_imagenet2012) else len(_ds_imagenet2012)
    _indices = torch.randperm(len(_ds_imagenet2012))[:_num_calibration] + 1  # +1是因为randperm生成0-999
    _subsets_imagenet2012 = torch.utils.data.Subset(_ds_imagenet2012, _indices)

    _loader_imagenet2012 = DataLoader(
        dataset=_subsets_imagenet2012,        # 单样本数据集
        batch_size=100,   # 数据集批次大小
        shuffle=False,  # 是否随机洗牌数据集 
    )
    return _loader_imagenet2012

In [6]:
loader_imagenet2012 = load_data()
for x, y in loader_imagenet2012:
    print(x.shape, y.shape)
    break

torch.Size([100, 3, 224, 224]) torch.Size([100])


### (5) 模型准确率评估

In [7]:
def eval_model(model, loader, device="cpu"):
    import torch
    model.eval()
    model = model.to(device)
    _num_total = 0 
    _num_corre = 0
    for _x, _y in loader:
        _x = _x.to(device)
        _y = _y.to(device)
        _y_ = model(_x)
    
        _prob, _cls_id = torch.max(_y_, dim=1)
        _num_corre += (_cls_id == _y).sum().item()
        _num_total += len(_cls_id)
    
    _accu = _num_corre * 100.0 / _num_total
    return _accu

In [8]:
accu = eval_model(model_w8a8, loader_imagenet2012)
print(F"准确率：{accu:.2f}%")

准确率：51.00%


In [9]:
accu = eval_model(model_w8a8, loader_imagenet2012, device="cuda")
print(F"准确率：{accu:.2f}%")

准确率：51.00%


### (6) 模型大小评估

In [10]:
def get_model_size(model):
    import os
    import torch
    # 因为量化后模型无法通过parameters()来计算模型大小，只能通过保存的文件来获取大小。
    torch.save(model.state_dict(), "model.pth")
    _size = os.path.getsize("model.pth") / 1024**2
    os.remove("model.pth")
    return _size

In [11]:
model_alex_size =  get_model_size(model_alex)
model_w8a8_size =  get_model_size(model_w8a8)
print(F"模型大小减少：{model_alex_size / model_w8a8_size:.2f}x({model_alex_size:.2f}MB -> {model_w8a8_size:.2f}MB)")

模型大小减少：3.56x(233.09MB -> 65.41MB)


### (7) 模型推理时间评估

In [12]:
def model_infer_time(model, loader, device="cpu"):
    import time
    import torch
    # 可选（编译成机器码，建议在GPU使用）
    # _model = torch.compile(model, mode="max-autotune", fullgraph=True)
    _model = model
    _model.eval()
    _model.to(device)
    # 预热
    for _x, _ in loader:
        _x = _x.to(device)
        _model(_x)
        
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(10):   # 循环10次，用来计算平均时间
        for _x, _ in loader:
            _x = _x.to(device)
            _model(_x)
    torch.cuda.synchronize()
    _infer_time = time.time() - start
    return _infer_time

In [14]:
model_alex_time = model_infer_time(model_alex, loader_imagenet2012, "cuda")
model_w8a8_time = model_infer_time(model_w8a8, loader_imagenet2012, "cuda")
print(F"推理时间提升：{model_alex_time / model_w8a8_time:.2f}x({model_alex_time:.2f}秒 -> {model_w8a8_time:.2f}秒)")

推理时间提升：0.86x(49.25秒 -> 57.13秒)


In [15]:
model_alex_time = model_infer_time(model_alex, loader_imagenet2012, "cpu")
model_w8a8_time = model_infer_time(model_w8a8, loader_imagenet2012, "cpu")
print(F"推理时间提升：{model_alex_time / model_w8a8_time:.2f}x({model_alex_time:.2f}秒 -> {model_w8a8_time:.2f}秒)")

推理时间提升：1.05x(86.17秒 -> 82.06秒)


## 2.2. 静态量化(训练后静态量化PTSQ)

- 静态量化是指在推理或生成过程中对所有输入使用固定的量化范围。
    - 与动态量化（为每个新的输入批次动态计算新的量化范围）不同，静态量化通常能实现更高效的计算，但可能牺牲一定的量化精度，因为它无法实时适应输入分布的变化。
    - 在静态量化中，这个固定的量化范围通常在量化模型之前，通过对类似的输入进行校准来确定。
    - 在校准阶段，我们首先在模型中插入观察器，以“观察”待量化输入的分布，然后利用这个分布来决定最终量化模型时要使用的缩放因子和零点。

- torchao 自带一个简单的观察器实现 AffineQuantizedMinMaxObserver，它会在校准阶段记录流经观察器的最小值和最大值。开发者可以实现自己所需、更高级的观察技术，例如那些依赖于移动平均或直方图的技术。这些技术未来可能会被添加到torchao中。
    - 在写本内容的时候，torchao实现的观察器只有AffineQuantizedMinMaxObserver可用，其他的还有：
        - AffineQuantizedFixedQParamObserver：允许手动设置固定量化参数的观察器。
        - AffineQuantizedMSEObserver：通过线性搜索来最小化由离群值引起的量化损失。

### (1) 加载模型

In [1]:
def load_model(model_name="alex"):
    import torch
    import torchvision
    if model_name == "alex":
        _model = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.DEFAULT)
        return _model
    if model_name == "vgg":
        _model = torchvision.models.vgg11(weights=torchvision.models.VGG11_Weights.DEFAULT)
        return _model
    if model_name == "resnet":
        _model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        return _model
    if model_name == "vit":
        _model = torchvision.models.vit_b_16(weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
        return _model

In [2]:
model_alex = load_model()
model_alex

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

### (2) 模型静态量化

- (a) 定义需要插入的观察器
    - 目前可用观察器只有AffineQuantizedMinMaxObserver。
    - 插入观察器的目的是校准量化数据参数。核心参数通过min与max值来计算出来的scale与zero_point。
    - 观察器定义的时候，需要设置的参数：
        - 量化的映射类型，由MappingType定义的枚举类型：（负责在fp32⇆int8类型之间转换）
            - SYMMETRIC：对称，比如适合torch.int8
            - SYMMETRIC_NO_CLIPPING_ERR：对称无裁剪误差
            - ASYMMETRIC：非对称,比如适合torch.uint8
        - 量化类型target_dtype，torch中可用的类型如下（并不是每个类型都在量化中可用）：
            - float32
            - float
            - float64
            - double
            - float16
            - bfloat16
            - float8_e4m3fn
            - float8_e4m3fnuz
            - float8_e5m2
            - float8_e5m2fnuz
            - float8_e8m0fnu
            - float4_e2m1fn_x2
            - half
            - uint8
            - uint16
            - uint32
            - uint64
            - int8
            - int16
            - short
            - int32
            - int
            - int64
            - long
            - complex32
            - complex64
            - chalf
            - cfloat
            - complex128
            - cdouble
            - quint8
            - qint8
            - qint32
            - bool
            - quint4x2
            - quint2x4
            - bits1x8
            - bits2x4
            - bits4x2
            - bits8
            - bits16
        - 粒度：
            - PerTensor()：
            - PerAxis(axis: int)：
            - PerGroup(group_size: int)：
            - PerRow(dim: int = -1)：
            - PerToken()：
            - PerBlock(block_size: tuple`[int, ...]`)：

In [3]:
def define_observer():
    import torch
    from torchao.quantization.granularity import PerAxis, PerTensor
    from torchao.quantization.observer import AffineQuantizedMinMaxObserver
    from torchao.quantization.quant_primitives import MappingType
    
    # per tensor input activation asymmetric quantization
    _active_obs = AffineQuantizedMinMaxObserver(
        MappingType.ASYMMETRIC,
        torch.uint8,
        granularity=PerTensor(),
        eps=torch.finfo(torch.float32).eps,
        scale_dtype=torch.float32,
        zero_point_dtype=torch.float32,
    )
    
    # per channel weight asymmetric quantization
    _weight_obs = AffineQuantizedMinMaxObserver(
        MappingType.ASYMMETRIC,
        torch.uint8,
        granularity=PerAxis(axis=0),
        eps=torch.finfo(torch.float32).eps,
        scale_dtype=torch.float32,
        zero_point_dtype=torch.float32,
    )

    return _active_obs, _weight_obs

In [4]:
active_obs, weight_obs = define_observer()
active_obs, weight_obs

W0317 16:55:00.724000 17544 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


(AffineQuantizedMinMaxObserver(), AffineQuantizedMinMaxObserver())

- (b) 定义被观察的模块/层
    - 常见的模块就是nn.Linear与nn.Conv1d/2d/3d。
    - 插入了上述观察器，是用于在校准期间记录输入激活值和权重的值。
    - 被观察的模块实现需要有如下几个规则：
        - 构造器参数在保持原Layer的基础上，增加激活层观察器与权重观察器。
            - 原构造器：`torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None)`
            - 被观察构造器：`torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None, act_obs: torch.nn.Module, weight_obs: torch.nn.Module)` 
        - 调用父类构造器：`super().__init__(in_features, out_features, bias, device, dtype)`
        - 必须实现from_float类函数，用于完成从原layer参数的移植。

In [5]:
import torch
class ObservedLinear(torch.nn.Linear):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        act_obs: torch.nn.Module,
        weight_obs: torch.nn.Module,
        bias: bool = True,
        device=None,
        dtype=None,
    ):
        super().__init__(in_features, out_features, bias, device, dtype)
        self.act_obs = act_obs
        self.weight_obs = weight_obs

    def forward(self, input: torch.Tensor):
        observed_input = self.act_obs(input)
        observed_weight = self.weight_obs(self.weight)
        return torch.nn.functional.linear(observed_input, observed_weight, self.bias)

    @classmethod
    def from_float(cls, float_linear, act_obs, weight_obs):
        observed_linear = cls(
            float_linear.in_features,
            float_linear.out_features,
            act_obs,
            weight_obs,
            False,
            device=float_linear.weight.device,
            dtype=float_linear.weight.dtype,
        )
        observed_linear.weight = float_linear.weight   # 值移植
        observed_linear.bias = float_linear.bias       # 值移植
        return observed_linear

- (c) 把原模型中对应的层替换成有观察器的层

In [6]:
def insert_observers(model, act_obs, weight_obs):
    import copy
    from torchao.quantization.quant_api import _replace_with_custom_fn_if_matches_filter
    _is_linear = lambda m, fqn: isinstance(m, torch.nn.Linear)

    def replacement_fn(m):
        _copied_act_obs = copy.deepcopy(act_obs)
        _copied_weight_obs = copy.deepcopy(weight_obs)
        return ObservedLinear.from_float(m, _copied_act_obs, _copied_weight_obs)

    _replace_with_custom_fn_if_matches_filter(model, replacement_fn, _is_linear)


In [7]:
import copy
model_static = copy.deepcopy(model_alex)
insert_observers(model_static, active_obs, weight_obs)

In [8]:
model_static

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): ObservedLinear(
      in_features=9216, out_features=409

### (3) 加载校准数据集（也可以用于校验模型）

- (a) 加载数据集

In [9]:
# 返回DataLoader对象。
def load_data(root="F:/04Datasets/ImageNet2012", split="val"):
    import torchvision.transforms as transforms
    from torchvision.datasets import ImageNet
    from torch.utils.data import DataLoader
    import torch
    import torchvision
    torch.manual_seed(2026)
    # 加载数据集
    _ds_imagenet2012 = ImageNet(
        root=root,
        split=split,
        transform = torchvision.models.AlexNet_Weights.IMAGENET1K_V1.transforms() # 需要是对象
        # target_transform=None,   # 标签转换
        # loader=Image.open   # 默认（还可以直接加载为Tensor：）
    )
    # 取部分子集
    _num_calibration = 1000   # 总样本是50000 
    _num_calibration = _num_calibration if _num_calibration<=len(_ds_imagenet2012) else len(_ds_imagenet2012)
    _indices = torch.randperm(len(_ds_imagenet2012))[:_num_calibration] + 1  # +1是因为randperm生成0-999
    _subsets_imagenet2012 = torch.utils.data.Subset(_ds_imagenet2012, _indices)

    _loader_imagenet2012 = DataLoader(
        dataset=_subsets_imagenet2012,        # 单样本数据集
        batch_size=100,   # 数据集批次大小
        shuffle=False,  # 是否随机洗牌数据集 
    )
    return _loader_imagenet2012

In [10]:
loader_imagenet2012 = load_data()

- (b) 校准模型

In [11]:
def calibration_model(model, loader, device="cpu"):
    model.to(device)
    for _ in range(1):  # 可以指定校准轮数
        for _x, _y in loader:
            model(_x)

In [12]:
calibration_model(model_static, loader_imagenet2012)
model_static

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): ObservedLinear(
      in_features=9216, out_features=409

### (4) 模型量化

- (a) 定义量化层
    - 调用观察器计算scale与zero_point。
    - 调用to_affine_quantized_intx_static函数实现具体的量化（数据转换）。

In [13]:
from torchao.dtypes import to_affine_quantized_intx_static
import torch
class QuantizedLinear(torch.nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        act_obs: torch.nn.Module,
        weight_obs: torch.nn.Module,
        weight: torch.Tensor,
        bias: torch.Tensor,
        target_dtype: torch.dtype,
    ):
        super().__init__()
        self.act_scale, self.act_zero_point = act_obs.calculate_qparams()
        weight_scale, weight_zero_point = weight_obs.calculate_qparams()
        assert weight.dim() == 2
        block_size = (1, weight.shape[1])
        self.target_dtype = target_dtype
        self.bias = bias
        self.qweight = to_affine_quantized_intx_static(
            weight, weight_scale, weight_zero_point, block_size, self.target_dtype
        )

    def forward(self, input: torch.Tensor):
        block_size = input.shape
        qinput = to_affine_quantized_intx_static(
            input,
            self.act_scale,
            self.act_zero_point,
            block_size,
            self.target_dtype,
        )
        return torch.nn.functional.linear(qinput, self.qweight, self.bias)

    @classmethod
    def from_observed(cls, observed_linear, target_dtype):  # 完成量化层的构造
        quantized_linear = cls(
            observed_linear.in_features,
            observed_linear.out_features,
            observed_linear.act_obs,
            observed_linear.weight_obs,
            observed_linear.weight,
            observed_linear.bias,
            target_dtype,
        )
        return quantized_linear

- (b) 定义量化配置
    - 这个配置最终被quantize_函数调用，用来进行量化。

In [14]:
from dataclasses import dataclass

from torchao.core.config import AOBaseConfig
from torchao.quantization import quantize_
from torchao.quantization.transform_module import register_quantize_module_handler

@dataclass
class StaticQuantConfig(AOBaseConfig):   # 静态量化配置
    target_dtype: torch.dtype

# 注册量化模块的处理器
@register_quantize_module_handler(StaticQuantConfig)
def _apply_static_quant(module: torch.nn.Module, config: StaticQuantConfig): 
    # 返回量化层
    return QuantizedLinear.from_observed(module, config.target_dtype)

# 判断层是否被量化
is_observed_linear = lambda m, fqn: isinstance(m, ObservedLinear)

# 执行静态量化
quantize_(model_static, StaticQuantConfig(torch.uint8), is_observed_linear)

In [15]:
model_static

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): QuantizedLinear()
    (2): ReLU(inplace=True)
    (3): D

In [16]:
model_static.classifier[1].act_scale, model_static.classifier[1].act_zero_point

(tensor(0.4012), tensor(0.))

In [17]:
model_static.classifier[1].qweight

AffineQuantizedTensor(tensor_impl=PlainAQTTensorImpl(data=tensor([[142, 149, 172,  ..., 128, 122,  99],
        [113, 115, 118,  ..., 126, 109,  58],
        [ 93, 105,  89,  ..., 146, 143, 203],
        ...,
        [114, 145, 107,  ..., 109, 155, 159],
        [ 78,  99,  86,  ..., 105, 102,  83],
        [138,  68, 120,  ..., 147, 148, 109]], dtype=torch.uint8)... , scale=tensor([0.0003, 0.0003, 0.0003,  ..., 0.0003, 0.0003, 0.0003])... , zero_point=tensor([129., 108., 147.,  ..., 136., 119., 127.])... , _layout=PlainLayout()), block_size=(1, 9216), shape=torch.Size([4096, 9216]), device=cpu, dtype=torch.float32, requires_grad=False)

### (4) 评估模型的准确率

In [18]:
def eval_model(model, loader, device="cpu"):
    import torch
    model.eval()
    model = model.to(device)
    _num_total = 0 
    _num_corre = 0
    for _x, _y in loader:
        _x = _x.to(device)
        _y = _y.to(device)
        _y_ = model(_x)
    
        _prob, _cls_id = torch.max(_y_, dim=1)
        _num_corre += (_cls_id == _y).sum().item()
        _num_total += len(_cls_id)
    
    _accu = _num_corre * 100.0 / _num_total
    return _accu

In [19]:
o_accu = eval_model(model_alex, loader_imagenet2012)
q_accu = eval_model(model_static, loader_imagenet2012)
print(F"原模型准确率：{o_accu:.2f}%，量化模型的准确率：{q_accu:.2f}%")

原模型准确率：51.20%，量化模型的准确率：51.30%


### (5) 评估模型大小

In [20]:
def get_model_size(model):
    import os
    import torch
    # 因为量化后模型无法通过parameters()来计算模型大小，只能通过保存的文件来获取大小。
    torch.save(model.state_dict(), "model.pth")
    _size = os.path.getsize("model.pth") / 1024**2
    os.remove("model.pth")
    return _size

In [22]:
o_size = get_model_size(model_alex)
q_size = get_model_size(model_static)
print(F"量化后模型大小减少：{o_size / q_size:.2f}x({o_size:.2f}MB -> {q_size:.2f}MB)")

量化后模型大小减少：24.64x(233.09MB -> 9.46MB)


### (6) 评估模型推理时间

In [26]:
def model_infer_time(model, loader, device="cpu"):
    import time
    import torch
    # 可选（编译成机器码，建议在GPU使用）
    # _model = torch.compile(model, mode="max-autotune", fullgraph=True)
    _model = model
    _model.eval()
    _model.to(device)
    # 预热
    for _x, _ in loader:
        _x = _x.to(device)
        _model(_x)
        
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(10):   # 循环10次，用来计算平均时间
        for _x, _ in loader:
            _x = _x.to(device)
            _model(_x)
    torch.cuda.synchronize()
    _infer_time = time.time() - start
    return _infer_time / 10

In [27]:
o_time = model_infer_time(model_alex, loader_imagenet2012, "cpu")
q_time = model_infer_time(model_static, loader_imagenet2012, "cpu")
print(F"推理时间提升：{o_time / q_time:.2f}x({o_time:.2f}秒 -> {q_time:.2f}秒)")

推理时间提升：0.92x(8.08秒 -> 8.81秒)


----

## 2.3. 训练感知量化(QAT)